In [0]:

from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

snapshot_root = (
    "abfss://curated@stflight60106354.dfs.core.windows.net/"
    "flight_delays/phase2/phase2_20251118_v1/"
)

train = spark.read.parquet(snapshot_root + "train/")
validation = spark.read.parquet(snapshot_root + "validation/")

categorical_cols = [
    "Reporting_Airline", "Origin", "Dest"
]

numeric_cols = [
    "Month", "CRSDepTime", "DepHour",
    "DayOfWeek", "IsWeekend", "Quarter"
]

feature_cols = categorical_cols + numeric_cols
target_col = "ArrDel15"

assert target_col not in feature_cols
assert "ArrDelay" not in feature_cols
assert all(c in train.columns for c in feature_cols)

print("Scikit-learn version:", sklearn.__version__)
print("Input features:", feature_cols)
print("Target:", target_col)
print("PASS: Model input features verified")


Scikit-learn version: 1.6.1
Input features: ['Reporting_Airline', 'Origin', 'Dest', 'Month', 'CRSDepTime', 'DepHour', 'DayOfWeek', 'IsWeekend', 'Quarter']
Target: ArrDel15
PASS: Model input features verified


In [0]:

# Deterministic sampling from fixed chronological snapshots.
# The final test dataset is not accessed here.

hash_columns = [
    "FlightDate",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "CRSDepTime"
]

def sample_flights(dataset, divisor):
    return dataset.filter(
        F.pmod(
            F.xxhash64(*hash_columns),
            F.lit(divisor)
        ) == 0
    )

train_sample = sample_flights(train, 50)        # About 2%
val_sample = sample_flights(validation, 20)     # About 5%

train_sample_count = train_sample.count()
val_sample_count = val_sample.count()

print("MODEL DEVELOPMENT SAMPLE")
print("Training sample:", train_sample_count)
print("Validation sample:", val_sample_count)

assert train_sample_count > 10000
assert val_sample_count > 10000

# Convert only the small samples to pandas.
train_pd = train_sample.select(
    *feature_cols, target_col
).toPandas()

val_pd = val_sample.select(
    *feature_cols, target_col
).toPandas()

X_train = train_pd[feature_cols].copy()
y_train = train_pd[target_col].astype(int)

X_val = val_pd[feature_cols].copy()
y_val = val_pd[target_col].astype(int)

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Training delay rate:", round(y_train.mean() * 100, 2), "%")
print("Validation delay rate:", round(y_val.mean() * 100, 2), "%")

print("PASS: Reproducible samples prepared")
print("PASS: Final test period remains untouched")


MODEL DEVELOPMENT SAMPLE
Training sample: 91931
Validation sample: 57588
Training rows: 91931
Validation rows: 57588
Training delay rate: 22.79 %
Validation delay rate: 18.37 %
PASS: Reproducible samples prepared
PASS: Final test period remains untouched


In [0]:

# Baseline on the same validation sample.
baseline = DummyClassifier(
    strategy="most_frequent"
)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_val)

# Preprocessing: fit category encoding on training data only.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        ),
        ("numeric", "passthrough", numeric_cols)
    ]
)

# Random Forest candidate.
rf = RandomForestClassifier(
    n_estimators=60,
    max_depth=12,
    min_samples_leaf=15,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=2
)

rf_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("classifier", rf)
])

print("Training Random Forest...")
rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_val)
rf_prob = rf_pipeline.predict_proba(X_val)[:, 1]

def evaluate(name, actual, predicted):
    return {
        "Model": name,
        "Accuracy": round(accuracy_score(actual, predicted), 4),
        "BalancedAccuracy": round(
            balanced_accuracy_score(actual, predicted), 4
        ),
        "Precision": round(
            precision_score(actual, predicted, zero_division=0), 4
        ),
        "Recall": round(
            recall_score(actual, predicted, zero_division=0), 4
        ),
        "F1": round(
            f1_score(actual, predicted, zero_division=0), 4
        )
    }

comparison = pd.DataFrame([
    evaluate("Baseline", y_val, baseline_pred),
    evaluate("Random Forest", y_val, rf_pred)
])

print("VALIDATION MODEL COMPARISON")
display(comparison)

print("Random Forest ROC-AUC:",
      round(roc_auc_score(y_val, rf_prob), 4))

print("Random Forest confusion matrix:")
print(confusion_matrix(y_val, rf_pred))

print("PASS: Random Forest training completed")
print("PASS: Both models evaluated on the same validation sample")


/databricks/python/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Training Random Forest...
VALIDATION MODEL COMPARISON


Model,Accuracy,BalancedAccuracy,Precision,Recall,F1
Baseline,0.8163,0.5,0.0,0.0,0.0
Random Forest,0.5261,0.6025,0.2389,0.7232,0.3592


Random Forest ROC-AUC: 0.6461
Random Forest confusion matrix:
[[22647 24364]
 [ 2928  7649]]
PASS: Random Forest training completed
PASS: Both models evaluated on the same validation sample


In [0]:

# STEP 4 — Tune the classification threshold
# Use validation data only. Do not access the final test set.

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_score, recall_score, f1_score
)

threshold_results = []

for threshold in np.arange(0.20, 0.81, 0.05):
    pred = (rf_prob >= threshold).astype(int)

    threshold_results.append({
        "Threshold": round(float(threshold), 2),
        "Accuracy": round(accuracy_score(y_val, pred), 4),
        "BalancedAccuracy": round(
            balanced_accuracy_score(y_val, pred), 4
        ),
        "Precision": round(
            precision_score(y_val, pred, zero_division=0), 4
        ),
        "Recall": round(
            recall_score(y_val, pred, zero_division=0), 4
        ),
        "F1": round(
            f1_score(y_val, pred, zero_division=0), 4
        )
    })

threshold_df = pd.DataFrame(threshold_results)

display(threshold_df)

# Select the threshold maximizing validation F1.
best_row = threshold_df.sort_values(
    ["F1", "Threshold"],
    ascending=[False, True]
).iloc[0]

best_threshold = float(best_row["Threshold"])

print("BEST VALIDATION THRESHOLD:", best_threshold)
print("Selected validation F1:", best_row["F1"])
print("Selected balanced accuracy:", best_row["BalancedAccuracy"])

print("PASS: Threshold selected using validation data only")


Threshold,Accuracy,BalancedAccuracy,Precision,Recall,F1
0.2,0.1837,0.5,0.1837,1.0,0.3103
0.25,0.1837,0.5,0.1837,1.0,0.3103
0.3,0.1837,0.5,0.1837,1.0,0.3103
0.35,0.1846,0.5004,0.1838,0.9996,0.3105
0.4,0.2727,0.5394,0.1969,0.961,0.3268
0.45,0.4773,0.5966,0.2298,0.7852,0.3556
0.5,0.5261,0.6025,0.2389,0.7232,0.3592
0.55,0.6174,0.6074,0.261,0.5916,0.3622
0.6,0.7951,0.5308,0.3309,0.1131,0.1686
0.65,0.8163,0.5,0.0,0.0,0.0


BEST VALIDATION THRESHOLD: 0.55
Selected validation F1: 0.3622
Selected balanced accuracy: 0.6074
PASS: Threshold selected using validation data only


In [0]:

from sklearn.metrics import confusion_matrix, classification_report

rf_selected_pred = (
    rf_prob >= best_threshold
).astype(int)

print("RANDOM FOREST — SELECTED THRESHOLD")
print("Threshold:", best_threshold)

print("\nClassification report:")
print(classification_report(
    y_val,
    rf_selected_pred,
    target_names=["On time", "Delayed"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_val, rf_selected_pred))

selected_metrics = evaluate(
    "Random Forest (selected threshold)",
    y_val,
    rf_selected_pred
)

display(pd.DataFrame([selected_metrics]))

print("PASS: Selected threshold evaluated")
print("PASS: Final holdout data remains untouched")


RANDOM FOREST — SELECTED THRESHOLD
Threshold: 0.55

Classification report:
              precision    recall  f1-score   support

     On time       0.87      0.62      0.73     47011
     Delayed       0.26      0.59      0.36     10577

    accuracy                           0.62     57588
   macro avg       0.57      0.61      0.54     57588
weighted avg       0.76      0.62      0.66     57588

Confusion matrix:
[[29297 17714]
 [ 4320  6257]]


Model,Accuracy,BalancedAccuracy,Precision,Recall,F1
Random Forest (selected threshold),0.6174,0.6074,0.261,0.5916,0.3622


PASS: Selected threshold evaluated
PASS: Final holdout data remains untouched


In [0]:

# STEP 5 — Validation error analysis by airline

error_df = val_pd.copy()

error_df["Actual"] = y_val.to_numpy()
error_df["Predicted"] = rf_selected_pred
error_df["DelayScore"] = rf_prob

error_df["Correct"] = (
    error_df["Actual"] == error_df["Predicted"]
).astype(int)

error_df["FalsePositive"] = (
    (error_df["Actual"] == 0)
    & (error_df["Predicted"] == 1)
).astype(int)

error_df["FalseNegative"] = (
    (error_df["Actual"] == 1)
    & (error_df["Predicted"] == 0)
).astype(int)

airline_errors = (
    error_df.groupby("Reporting_Airline")
    .agg(
        Flights=("Actual", "size"),
        ActualDelayRate=("Actual", "mean"),
        Accuracy=("Correct", "mean"),
        FalsePositives=("FalsePositive", "sum"),
        FalseNegatives=("FalseNegative", "sum")
    )
    .reset_index()
)

# Focus on airlines with enough validation examples.
airline_errors = airline_errors[
    airline_errors["Flights"] >= 100
].copy()

airline_errors["ActualDelayRate"] = (
    airline_errors["ActualDelayRate"] * 100
).round(2)

airline_errors["Accuracy"] = (
    airline_errors["Accuracy"] * 100
).round(2)

airline_errors = airline_errors.sort_values(
    "Accuracy", ascending=True
)

print("VALIDATION ERROR ANALYSIS BY AIRLINE")
display(airline_errors)

print("\nOverall false positives:",
      error_df["FalsePositive"].sum())

print("Overall false negatives:",
      error_df["FalseNegative"].sum())

print("PASS: Validation error analysis completed")


VALIDATION ERROR ANALYSIS BY AIRLINE


Reporting_Airline,Flights,ActualDelayRate,Accuracy,FalsePositives,FalseNegatives
F9,1526,20.71,53.67,597,110
OH,1969,17.06,54.8,775,115
AA,8048,21.88,56.15,2966,563
MQ,2620,16.53,56.91,983,146
G4,838,21.12,57.52,310,46
NK,1462,16.28,57.93,520,95
AS,2051,18.28,58.22,663,194
YX,2818,21.86,60.08,911,214
B6,1689,24.04,60.33,551,119
UA,6808,18.08,60.53,2226,461



Overall false positives: 17714
Overall false negatives: 4320
PASS: Validation error analysis completed


In [0]:

import mlflow
import mlflow.sklearn
import inspect
import numpy as np

from mlflow.models import infer_signature

# Save the existing fitted Random Forest pipeline.
# This includes its fitted categorical preprocessing.

assert hasattr(rf_pipeline, "predict_proba")
assert len(X_train) > 0
assert len(X_val) > 0

example = X_train.head(5).copy()

signature = infer_signature(
    example,
    rf_pipeline.predict_proba(example)
)

with mlflow.start_run(
    run_name="Flight_Delay_RF_Phase2"
) as run:

    mlflow.log_param(
        "dataset_version", "phase2_20251118_v1"
    )
    mlflow.log_param(
        "training_sample_size", int(len(X_train))
    )
    mlflow.log_param(
        "validation_sample_size", int(len(X_val))
    )
    mlflow.log_param(
        "random_state", 42
    )
    mlflow.log_param(
        "selected_threshold", float(best_threshold)
    )

    mlflow.set_tags({
        "project": "DSAI3202_Flight_Delay",
        "model_type": "RandomForest",
        "training_period": "2025-01_to_2025-08",
        "validation_period": "2025-09_to_2025-10",
        "sample_rule_train": "xxhash64_mod_50_eq_0",
        "sample_rule_validation": "xxhash64_mod_20_eq_0"
    })

    for metric, value in selected_metrics.items():
        if metric != "Model":
            mlflow.log_metric(
                "validation_" + metric.lower(),
                float(value)
            )

    # Support MLflow 2 and MLflow 3.
    model_args = {
        "sk_model": rf_pipeline,
        "signature": signature,
        "input_example": example
    }

    parameters = inspect.signature(
        mlflow.sklearn.log_model
    ).parameters

    if "name" in parameters:
        model_args["name"] = "flight_delay_rf_candidate"
    else:
        model_args["artifact_path"] = "model"

    model_info = mlflow.sklearn.log_model(
        **model_args
    )

    saved_run_id = run.info.run_id
    saved_model_uri = model_info.model_uri

print("PASS: Model saved using MLflow")
print("Run ID:", saved_run_id)
print("Model URI:", saved_model_uri)
print("Selected threshold:", best_threshold)


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://adb-7405613772250173.13.azuredatabricks.net/ml/experiments/4134578384466741/models/m-ef8f1d44e4d0475b9dfead71d2f4b78d?o=7405613772250173


PASS: Model saved using MLflow
Run ID: 3b5c8e7ea8ce490da766182e8db9f04b
Model URI: models:/m-ef8f1d44e4d0475b9dfead71d2f4b78d
Selected threshold: 0.55


In [0]:

# Load the model from MLflow and compare predictions.

restored_model = mlflow.sklearn.load_model(
    saved_model_uri
)

original_scores = rf_pipeline.predict_proba(
    X_val.iloc[:20]
)[:, 1]

restored_scores = restored_model.predict_proba(
    X_val.iloc[:20]
)[:, 1]

np.testing.assert_allclose(
    original_scores,
    restored_scores,
    rtol=0,
    atol=1e-12
)

print("PASS: Model loaded successfully from MLflow")
print("PASS: Predictions match the original model")
print("Run ID:", saved_run_id)
print("Model URI:", saved_model_uri)
print("PASS: Model artifact verified")


PASS: Model loaded successfully from MLflow
PASS: Predictions match the original model
Run ID: 3b5c8e7ea8ce490da766182e8db9f04b
Model URI: models:/m-ef8f1d44e4d0475b9dfead71d2f4b78d
PASS: Model artifact verified


In [0]:

import mlflow
import pandas as pd
import numpy as np

# Load the trained Random Forest model
model_uri = "models:/m-ef8f1d44e4d0475b9dfead71d2f4b78d"
model = mlflow.sklearn.load_model(model_uri)

# Same flight used in the Azure ML online endpoint
flight = pd.DataFrame([{
    "Reporting_Airline": "AA",
    "Origin": "RDU",
    "Dest": "PHL",
    "Month": 8,
    "CRSDepTime": 1610,
    "DepHour": 16,
    "DayOfWeek": 7,
    "IsWeekend": 1,
    "Quarter": 3
}])

threshold = 0.55

# Offline prediction
offline_probability = float(model.predict_proba(flight)[0, 1])
offline_prediction = int(offline_probability >= threshold)

# Actual Azure ML online prediction
online_probability = 0.606826418671098
online_prediction = 1

difference = abs(offline_probability - online_probability)

print("OFFLINE VS ONLINE CONSISTENCY TEST")
print("----------------------------------")
print("Offline probability:", offline_probability)
print("Online probability: ", online_probability)
print("Offline prediction: ", offline_prediction)
print("Online prediction:  ", online_prediction)
print("Probability difference:", difference)

assert np.isclose(
    offline_probability,
    online_probability,
    atol=1e-6,
    rtol=0
), "Offline and online probabilities do not match"

assert offline_prediction == online_prediction, \
    "Offline and online predictions do not match"

print("PASS: Offline and online predictions are consistent")


OFFLINE VS ONLINE CONSISTENCY TEST
----------------------------------
Offline probability: 0.606826418671098
Online probability:  0.606826418671098
Offline prediction:  1
Online prediction:   1
Probability difference: 0.0
PASS: Offline and online predictions are consistent
